# Disco completo GOES-East (G19) · infrarrojo limpio C13 (10.3 µm)

Animación del **disco completo** con archivos reales
`OR_ABI-L2-CMIPF-M6C13_G19_*.nc` (NOAA/NESDIS, bucket público `noaa-goes19`).

- **Satélite y banda**: GOES-East (G19, en 75.2°O), banda **C13 (10.33 µm)**,
  temperatura de brillo en °C. Es la banda "Clean Infrared": ve nubes de día y de noche.
- **Encuadre**: todo lo que abarca el dataset (disco completo, sin zoom).
- **Escala**: percentiles automáticos sobre el primer cuadro, igual que `poster_goes.py`;
  paleta `termico` (frío = blanco, como en las imágenes IR de siempre).
- **Cadencia**: 10 min (modo 6); el nombre del archivo la trae escrita.
- **Vestuario**: el mismo de `poster_goes.py` (`draw_goes_poster` + `caption_lines`).


In [ ]:
# ==== configuración ====
from datetime import datetime
from pathlib import Path

# Raíz del proyecto: el notebook vive en prototipos/notebooks/.
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "comun").is_dir())

T0       = datetime(2026, 10, 12, 21, 30)   # primer cuadro (UTC): 24 h hasta el ultimo dato
N_FRAMES = 144                              # 24 h a 10 min
PASO_MIN = 10                              # cadencia real de C13 en modo 6

SAT, BANDA = "G19", "C13"                  # GOES-East, IR limpio 10.3 µm
PALETA     = "ir_realce"                   # IR con realce de color
CREDITO    = "Graphic by @Gillo97"      # tu handle: arriba a la derecha + marca de agua

DESCARGAR  = False                          # False = solo archivos locales; falla si falta un cuadro

# Escala de la barra: None = percentiles automáticos, igual que poster_goes.py.
VMIN, VMAX = None, None
ESCALA_PERCENTILES = (0.5, 99.5)
TOPE_PX = 2712                             # 1:1 con el dato (5424 px en 2 km -> 2712 en 4 km)

LADO_PULG, DPI = 19.3714, 140              # 2712 px por cuadro (1:1 con el dato)
GEOJSON = str(RAIZ / "datos" / "paraguay_gadm.json")             # borde destacado (GADM, 1893 vertices)
CENTRO_LON, CENTRO_LAT = -73.0, -18.0       # Orthographic girado: encuadre aprobado
CUADRO_MS, FUNDIDO_MS, FUNDIDO_PASOS = 80, 30, 1
HACER_GIF, HACER_MP4 = False, True          # a esta resolucion el GIF seria ~1 GB
CRF_MP4, PRESET_MP4 = 16, "slow"           # CRF mas bajo = mas calidad; preset lento comprime mejor


In [ ]:
# ==== datos y utilidades ====
import gc, re, sys
from datetime import timedelta
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image as MostrarImagen, display
from rich.progress import track

sys.path.insert(0, str(RAIZ / "comun"))
sys.path.insert(0, str(RAIZ / "prototipos"))
sys.path.insert(0, str(RAIZ / "descargar"))
from poster_goes import BG, PALETTES, draw_goes_poster
from goes_lectura import cargar, cuadros_con_fundido
from goes_descarga import clave_de, descargar
from rutas import DATOS as RAIZ_DATOS, SALIDAS

DATOS = RAIZ_DATOS / f"goes_c{BANDA[1:].lower()}_{SAT.lower()}"
DATOS.mkdir(parents=True, exist_ok=True)
SALIDAS.mkdir(parents=True, exist_ok=True)
PX = int(round(LADO_PULG * DPI))   # lado en píxeles, va en el nombre de las salidas


def cargar_cuadro(nc):
    """Disco completo submuestreado a TOPE_PX, con los rótulos del póster."""
    return cargar(nc, tope_px=TOPE_PX, escena="FULL DISK")


SLOTS = [T0 + i * timedelta(minutes=PASO_MIN) for i in range(N_FRAMES)]
print(f"{len(SLOTS)} cuadros: {SLOTS[0]:%Y-%m-%d %H:%M} -> {SLOTS[-1]:%Y-%m-%d %H:%M} UTC")


In [ ]:
# ==== descarga (solo lo que falta; nada si DESCARGAR=False y todo está local) ====
rutas = []
faltan = []
for t in track(SLOTS, description="Descargando cuadros"):
    try:
        ruta = descargar(DATOS, SAT, clave_de(SAT, BANDA, t), DESCARGAR)
    except LookupError:
        print(f"  salto {t:%Y-%m-%d %H:%M} UTC (hueco en NOAA)")
        faltan.append(t)
        continue
    rutas.append(ruta)
    print(f"{t:%Y-%m-%d %H:%M} UTC  {ruta.name}  {ruta.stat().st_size / 1e6:.1f} MB")
print(f"listos: {len(rutas)}/{len(SLOTS)} | huecos: {len(faltan)}")


In [ ]:
# ==== control: rótulos del primer cuadro ====
campo, x, y, proj, unidad, rotulos = cargar_cuadro(rutas[0])
titulo, subtitulo, fuente, datos, tiempo = rotulos
if VMIN is None or VMAX is None:
    lo, hi = np.nanpercentile(campo, ESCALA_PERCENTILES)
    VMIN = VMIN if VMIN is not None else float(lo)
    VMAX = VMAX if VMAX is not None else float(hi)
    print(f"escala automática {ESCALA_PERCENTILES}: {VMIN:.2f} a {VMAX:.2f} {unidad}")
print(f"disco: {campo.shape} celdas")
print(titulo)
print(subtitulo)
print(fuente)
print(datos)
assert unidad in ("°C", "%")
assert re.match(r"^[A-Z]+ \d{4}$", titulo), titulo
assert SAT in subtitulo and SAT in fuente and SAT in datos
assert datos.startswith(f"DATA: {SAT} |") and datos.endswith("MODO Mode 6")
assert "NOAA/NESDIS" in fuente and "ABI L2 CMIP" in fuente
print("\nOK: rótulos verificados.")


In [ ]:
# ==== cuadro fijo (PNG) del primer instante ====
fig, ax, _ = draw_goes_poster(
    campo, x=x, y=y, proj=proj, title=titulo, subtitle=subtitulo, unit_label=unidad,
    src_line=fuente, data_line=datos, credit=CREDITO, time_line=tiempo,
    cmap=PALETTES[PALETA], vmin=VMIN, vmax=VMAX,
    highlight_geojson=GEOJSON, globe_center=(CENTRO_LON, CENTRO_LAT))
png = SALIDAS / f"poster_globo_{SAT}_{T0:%Y-%m-%dT%H%M}_{PX}px.png"
fig.savefig(png, facecolor=BG)
plt.close(fig)
print("PNG:", png)
display(MostrarImagen(filename=str(png)))


In [ ]:
# ==== render de todos los cuadros ====
# Si ya hay un cuadro por slot en frames_dir, se reutilizan (re-ejecutable sin re-renderizar).
frames_dir = SALIDAS / f"frames_{SAT}_{T0:%Y-%m-%dT%H%M}_{N_FRAMES}f_{PX}px"
frames_dir.mkdir(parents=True, exist_ok=True)
archivos = sorted(frames_dir.glob("cuadro_*.png"))

if len(archivos) == len(rutas):
    print(f"{len(archivos)} cuadros ya renderizados en {frames_dir} (se saltea el render)")
else:
    archivos = []
    for i, ruta in enumerate(track(rutas, description="Renderizando"), 1):
        campo_i, x_i, y_i, proj_i, unidad_i, rot_i = cargar_cuadro(ruta)
        fig, ax, _ = draw_goes_poster(
            campo_i, x=x_i, y=y_i, proj=proj_i, title=rot_i[0], subtitle=rot_i[1],
            unit_label=unidad_i, src_line=rot_i[2], data_line=rot_i[3],
            credit=CREDITO, time_line=rot_i[4], cmap=PALETTES[PALETA], vmin=VMIN, vmax=VMAX,
            size=LADO_PULG, dpi=DPI, highlight_geojson=GEOJSON,
            globe_center=(CENTRO_LON, CENTRO_LAT))
        fp = frames_dir / f"cuadro_{i:03d}.png"
        fig.savefig(fp, format="png", facecolor=BG)
        plt.close(fig)
        archivos.append(fp)
        del campo_i, x_i, y_i, fig, fp
        gc.collect()
        print(f"  cuadro {i}/{len(rutas)}  {rot_i[3]}")

print(f"{len(archivos)} cuadros en {frames_dir}")


In [ ]:
# ==== GIF con fundido entre cuadros ====
if HACER_GIF:
    fotogramas = list(cuadros_con_fundido(
        sorted(archivos), FUNDIDO_PASOS, CUADRO_MS, FUNDIDO_MS))
    gif = SALIDAS / f"animacion_globo_{SAT}_{T0:%Y-%m-%dT%H%M}_{N_FRAMES}f_{PX}px.gif"
    fotogramas[0][0].save(
        gif, save_all=True,
        append_images=[im for im, _ in fotogramas[1:]],
        duration=[ms for _, ms in fotogramas], loop=0, optimize=True)
    print(f"GIF: {gif}  ({len(fotogramas)} fotogramas, {gif.stat().st_size / 1e6:.1f} MB)")
    display(MostrarImagen(filename=str(gif)))
else:
    print("GIF desactivado (HACER_GIF=False): mirá el MP4 de la celda siguiente")


In [ ]:
# ==== MP4: misma animación en mucho menos peso y mejor calidad ====
# ffmpeg viene con imageio-ffmpeg, no hace falta que esté en el PATH.
import subprocess

import imageio_ffmpeg

if HACER_MP4:
    px = int(round(LADO_PULG * DPI))
    mp4 = SALIDAS / f"animacion_globo_{SAT}_{T0:%Y-%m-%dT%H%M}_{N_FRAMES}f_{px}px.mp4"
    proc = subprocess.run([
        imageio_ffmpeg.get_ffmpeg_exe(), "-y",
        "-framerate", f"{1000 / CUADRO_MS:.6f}",   # cada cuadro real dura CUADRO_MS
        "-start_number", "1",
        "-i", str(frames_dir / "cuadro_%03d.png"),
        # x264 con yuv420p exige lados pares: el pad ajusta 1 px sin remuestrear.
        "-vf", "pad=ceil(iw/2)*2:ceil(ih/2)*2",
        "-c:v", "libx264", "-preset", PRESET_MP4, "-pix_fmt", "yuv420p",
        "-crf", str(CRF_MP4),
        "-movflags", "+faststart", str(mp4),
    ], capture_output=True, text=True, encoding="utf-8", errors="replace")
    if proc.returncode:
        print(proc.stderr[-1500:])          # que se vea el motivo si falla
        raise RuntimeError(f"ffmpeg falló con código {proc.returncode}")
    print(f"MP4: {mp4}  ({mp4.stat().st_size / 1e6:.1f} MB, {px}x{px})")


## Ideas y variantes

- **Otra ventana**: cambiá `T0` y `N_FRAMES`; la clave S3 se arma sola a partir del
  slot de 10 minutos (no hace falta escribir el nombre del archivo).
- **Sin descargar**: `DESCARGAR = False` usa solo los .nc que ya estén en la carpeta
  (o en `goes_c13_g19/`); si falta un cuadro, avisa cuál en vez de bajarlo.
- **Zoom regional**: ver `animacion_paraguay_g19.ipynb` (misma estructura, recorte a la
  zona `py` a resolución completa).
- **Escala fija**: poné `VMIN`/`VMAX` en °C para no depender del primer cuadro.
- **Sin fundido**: `FUNDIDO_PASOS = 0` deja la animación a saltos limpios de 10 min.
- **MP4**: se genera con el ffmpeg que trae `imageio-ffmpeg` (~7x menos peso que
  el GIF y sin dithering). `HACER_GIF=False` saltea el GIF, que a 1176 px pesa mucho.
- **Crédito**: `CREDITO` alimenta el texto de arriba a la derecha y la marca de agua
  central (esa se dibuja con las letras separadas).
